# Refactor Validation — Phases 1-4

Validates that the 4-phase architecture decoupling works end-to-end:
1. **Shared utilities** — `_arun`, `is_option_symbol`, `update_candidate_from_snapshot`, `build_execution_components`
2. **Stepped executor** — `SteppedOrderExecutor` shared by CSP + CC
3. **Exit router** — `ExitRouter` shared by CSP + CC
4. **Market session** — `MarketSession` extracted from TradingLoop
5. **Position proxy** — `PositionProxy` replaces ad-hoc SimpleNamespace
6. **CSP cycle** — TradingLoop constructs and runs
7. **CC cycle** — CoveredCallManager constructs and runs

**Run during market hours for full coverage, or anytime for import/construction checks.**

In [1]:
# Universe screener (options liquidity)
from universe_screener.universe import get_combined_universe
from universe_screener.main import screen_tickers, get_next_fridays

# Alpaca client manager (Phase 2)
# Why is this in csp? It's own package. 
from csp.clients import AlpacaClientManager
from csp.data.options import OptionsDataFetcher

# Equity screener (Phase 1)
from equity_screener.config import EquityScreenerConfig
from equity_screener.main import run_screener

In [2]:
# Imports — all modules across Phases 1-4
from dotenv import load_dotenv
load_dotenv(override=True)

# Shared utilities (Phase 4)
from csp.trading.utils import _arun, is_option_symbol, update_candidate_from_snapshot, build_execution_components

# Extracted modules (Phases 2-3)
from csp.trading.stepped_executor import SteppedOrderExecutor
from csp.trading.exit_router import ExitRouter
from csp.trading.market_session import MarketSession
from csp.trading.models import PositionProxy, ExitReason, RiskCheckResult, OrderResult
from csp.trading.execution_config import ExecutionConfig, RiskConfig

# Core modules
from csp.config import StrategyConfig
from csp.data.vix import VixDataFetcher
from csp.data.greeks import GreeksCalculator
from csp.data.manager import DataManager
from csp.signals.scanner import StrategyScanner
from csp.trading.execution import ExecutionEngine
from csp.trading.risk import RiskManager
from csp.trading.metadata import StrategyMetadataStore
from csp.trading.loop import TradingLoop
from csp.trading.portfolio import print_portfolio_status
from csp.storage import build_storage_backend

# Equity screener (Phase 1)
from equity_screener.config import EquityScreenerConfig
from equity_screener.main import run_screener

# Universe screener (options liquidity)
from universe_screener.universe import get_combined_universe
from universe_screener.main import screen_tickers, get_next_fridays

# CC modules
from covered_call.config import CoveredCallConfig
from covered_call.store import WheelPositionStore
from covered_call.manager import CoveredCallManager

print("All imports OK")

All imports OK


## Universe Screener (Options Liquidity)

In [7]:
print('Universe Screener\n\n')
# Build ticker universe and screen for options liquidity
alpaca_screener = AlpacaClientManager(paper=True)
options_fetcher = OptionsDataFetcher(alpaca_screener)
screener_config = EquityScreenerConfig.from_name("csp_bullish")


universe = get_combined_universe(verbose=True)
results = screen_tickers(universe, options_fetcher, sample_size=100, verbose=True)

Universe Screener


Universe: S&P=501 | DJIA=30 | NASDAQ-100=101 | Combined=515
Target expirations: 2026-02-27 and 2026-03-06
Screening 100 tickers...

  Screened 25/100: 10 pass, 15 fail
  Screened 50/100: 21 pass, 29 fail
  Screened 75/100: 32 pass, 43 fail
  Screened 100/100: 42 pass, 58 fail

Done: 42 pass / 58 fail (of 100 screened)

Next Fridays: 2026-02-27 and 2026-03-06
Passing tickers (42): ['ACN', 'ADBE', 'ADI', 'AMGN', 'ANET', 'BX', 'C', 'CBOE', 'CCL', 'CDNS', 'COP', 'CRH', 'CTAS', 'CVS', 'DELL', 'DHR', 'DOW', 'EA', 'FCX', 'FISV', 'FSLR', 'FTNT', 'HPE', 'HRL', 'HSY', 'LRCX', 'LW', 'MA', 'MAR', 'MCD', 'NEE', 'NEM', 'OXY', 'PANW', 'PDD', 'PG', 'PLTR', 'STZ', 'VLO', 'VZ', 'WFC', 'XOM']
Failed tickers (58): ['LDOS', 'AIZ', 'MET', 'STX', 'VICI', 'PCAR', 'BDX', 'MTB', 'AMCR', 'ORLY', 'FFIV', 'RF', 'CCI', 'SWKS', 'COR', 'VTRS', 'NXPI', 'HUBB', 'JBL', 'CPRT', 'NOC', 'COST', 'COIN', 'SRE', 'VMC', 'KIM', 'MPWR', 'PLD', 'TROW', 'FITB', 'OTIS', 'JKHY', 'MTD', 'ELV', 'PNW', 'BKR', 'EPAM'

## Equity Screener (Technical Filter + Events)

In [8]:
# Run equity screener on tickers that passed the universe screener
screener_config = EquityScreenerConfig.from_name("csp_bullish")

equity_results = run_screener(
    screener_config,
    tickers=results['pass'],
    verbose=True,
)

Equity Screener: csp_bullish
  Universe: 42 tickers
  Fetching 60d price history...
  Got prices for 42 symbols
  Passed technical filter: 4/42
  No event rejections
  Saved to equity_screened_csp_bullish.json
  Pass: 4 | Fail: 38 | Events: 0

    Symbol         Price     SMA8    SMA20    SMA50    BB_Up    RSI Trend
    ------ ---- -------- -------- -------- -------- -------- ------ -----
    ACN       ✗   207.36   219.75   240.58   261.14   262.66   18.9     ↓
    ADBE      ✗   245.42   258.79   273.57   309.97   290.61   20.9     ↓
    ADI       ✗   355.28   343.17   327.01   302.69   342.53   78.8     ↑
    AMGN      ✓   380.62   373.26   362.84   342.70   377.76   68.5     ↑
    ANET      ✗   128.60   137.28   139.40   133.61   145.45   41.4     ↑
    BX        ✗   113.90   127.29   134.02   146.98   143.15   22.0     ↓
    C         ✗   110.12   113.89   116.07   116.32   119.67   41.7     ↑
    CBOE      ✗   292.93   281.37   274.45   264.70   283.14   75.9     ↑
    CCL       ✗ 

## Options Screener (Put Chain Filter)

Scans equity-passing tickers for put option contracts matching strategy criteria.
- `StrategyScanner.scan_tickers()` — fetches put chains, filters by delta/DTE/strike/premium, ranks candidates
- Uses default `StrategyConfig` filter params (delta=[0, 0.40], DTE=[1, 10], max_strike=97%)
- `skip_equity_filter=True` since tickers already passed the equity screener

In [ ]:
# Scan equity-passing tickers for put options
from csp.data.equity import EquityDataFetcher

options_config = StrategyConfig(paper_trading=True)
equity_fetcher = EquityDataFetcher(alpaca_screener)
greeks_calc = GreeksCalculator()

scanner = StrategyScanner(
    config=options_config,
    equity_fetcher=equity_fetcher,
    options_fetcher=options_fetcher,
    greeks_calc=greeks_calc,
    equity_screener_config=screener_config,
)

# Extract ticker symbols from equity screener output
equity_passing = [t['symbol'] for t in equity_results['pass']]

scan_results = scanner.scan_tickers(
    equity_passing,
    skip_equity_filter=True,
    verbose=True,
)

## CSP Entry

Pick a candidate from scan results and enter via the already-constructed `TradingLoop`.
- Change `idx` to select a different candidate

In [ ]:
# Collect best candidate per ticker (lowest strike = most conservative)
from itertools import groupby

all_candidates = [c for r in scan_results for c in r.options_candidates]
all_candidates.sort(key=lambda c: c.underlying)
best = []
for _, grp in groupby(all_candidates, key=lambda c: c.underlying):
    best.append(sorted(grp, key=lambda c: c.strike)[0])

for i, c in enumerate(best):
    drop = (c.stock_price - c.strike) / c.stock_price
    print(f"  [{i}] {c.underlying}: ${c.strike:.2f} ({drop:.1%} OTM) | "
          f"DTE={c.dte} | Bid=${c.bid:.2f} | Delta={abs(c.delta or 0):.3f}")

# ── Pick candidate and size ────────────────────────────────────
idx = 0
candidate = best[idx]
available = await _arun(alpaca.compute_available_capital)
collateral = candidate.collateral_required
qty = max(1, min(int(available * config.max_position_pct // collateral),
                 config.max_contracts_per_ticker))

print(f"\nEntering: {candidate.symbol} x{qty}")

# ── Execute via TradingLoop (handles order + metadata + logging)
success = await loop._execute_single_entry(
    candidate, qty, current_vix, index=1, total_count=1,
)

## Shared Utilities Smoke Test

In [ ]:
# Cell 2: Test is_option_symbol
assert not is_option_symbol("AAPL"), "Stock ticker should be False"
assert not is_option_symbol("MSFT"), "Stock ticker should be False"
assert is_option_symbol("AAPL260320P00220000"), "Option symbol should be True"
assert is_option_symbol("MSFT260320C00400000"), "Option symbol should be True"
print("is_option_symbol: PASS")

# Test update_candidate_from_snapshot
from types import SimpleNamespace
candidate = SimpleNamespace(bid=0, ask=0, mid=0, delta=0, implied_volatility=0)
update_candidate_from_snapshot(candidate, {"bid": 1.50, "ask": 2.00, "delta": -0.30})
assert candidate.bid == 1.50
assert candidate.ask == 2.00
assert candidate.mid == 1.75
assert candidate.delta == -0.30
print("update_candidate_from_snapshot: PASS")

# Test _arun
result = await _arun(lambda x, y: x + y, 2, 3)
assert result == 5
print("_arun: PASS")

print("\nAll utility tests passed")

In [ ]:
# Cell 3: Test PositionProxy
from datetime import date, timedelta

proxy = PositionProxy(
    symbol="AAPL",
    option_symbol="AAPL260320P00220000",
    quantity=-1,
    strike=220.0,
    expiration=date.today() + timedelta(days=5),
    entry_premium=1.50,
)

# P&L: sold at 1.50, buy back at 0.50 = $100 profit
pnl = proxy.calculate_pnl(0.50)
assert pnl == 100.0, f"Expected 100.0, got {pnl}"
print(f"PositionProxy P&L test: sold@1.50, buyback@0.50 → ${pnl:.2f} PASS")

# Test RiskConfig adapter
config_test = StrategyConfig(paper_trading=True)
risk_cfg = RiskConfig.from_strategy_config(config_test)
print(f"RiskConfig adapter: delta_stop={risk_cfg.enable_delta_stop} PASS")

# Test ExecutionConfig adapter
exec_cfg = ExecutionConfig.from_strategy_config(config_test)
print(f"ExecutionConfig adapter: order_type={exec_cfg.entry_order_type}, steps={exec_cfg.entry_max_steps} PASS")

print("\nAll model/config tests passed")

## Configuration

Edit each config cell below before running. Each config is its own dataclass.

In [ ]:
# ── Covered Call Config ──────────────────────────────────────────
cc_config = CoveredCallConfig(
    enabled=True,

    # DTE range
    cc_min_dte=1,
    cc_max_dte=10,

    # Strike selection: "delta" | "min_delta" | "pct_change" | "min_pct_change" | "min_daily_return"
    cc_strike_mode="delta",
    cc_strike_delta=0.30,              # used by "delta" and "min_delta"
    cc_strike_pct=0.02,                # used by "pct_change" and "min_pct_change" (2% OTM)
    cc_min_daily_return_pct=0.0015,    # used by "min_daily_return" (0.15%/day)

    # Chain fetch range (how wide to search for contracts)
    cc_min_strike_pct=1.01,            # ATM floor
    cc_max_strike_pct=1.15,            # 15% OTM ceiling

    # Diagnostics
    cc_verbose=False,
)

print(f"CoveredCallConfig: mode={cc_config.cc_strike_mode}, delta={cc_config.cc_strike_delta}, "
      f"DTE=[{cc_config.cc_min_dte}, {cc_config.cc_max_dte}]")

In [ ]:
# ── Strategy Config (CSP core) ──────────────────────────────────
config = StrategyConfig(
    # Capital & universe
    # ticker_universe=["AAPL", "MSFT", "GOOG"],  # default: S&P 500
    starting_cash=1_000_000,
    max_position_pct=0.10,                        # max 10% per ticker

    # Contract selection: "daily_return_per_delta" | "days_since_strike" |
    #   "daily_return_on_collateral" | "lowest_strike_price" | "lowest_delta"
    contract_rank_mode="lowest_strike_price",
    universe_rank_mode="lowest_delta",

    # Options filters
    delta_min=0.00,
    delta_max=0.40,
    min_dte=1,
    max_dte=10,
    min_daily_return=0.0015,
    max_strike_mode="pct",                        # "sma" or "pct"
    max_strike_pct=0.97,

    # Entry orders: "market" or "stepped"
    entry_order_type="market",
    entry_start_price="mid",                      # "mid" or "bid" (stepped only)
    entry_step_interval=3,                         # seconds between steps
    entry_step_pct=0.25,                           # fraction of spread per step
    entry_max_steps=4,

    # Exit orders (early exit / expiry — stops always use market)
    exit_start_price="mid",                        # "mid" or "ask"
    exit_step_interval=3,
    exit_step_pct=0.25,
    exit_max_steps=4,
    early_exit_buffer_pct=1.00,                    # exit when captured >= expected * (1 + buffer)

    # Risk management
    delta_stop_multiplier=2.0,                     # exit if delta >= 2x entry
    delta_absolute_stop=0.40,                      # exit if delta >= 0.40
    stock_drop_stop_pct=0.05,                      # exit if stock drops 5%
    vix_spike_multiplier=1.15,                     # exit if VIX >= 1.15x session open

    # Risk toggles
    enable_delta_stop=False,
    enable_delta_absolute_stop=True,
    enable_stock_drop_stop=False,
    enable_vix_spike_stop=True,
    enable_early_exit=True,

    # Operational
    paper_trading=True,
    print_mode="verbose",                          # "summary" or "verbose"

    # Covered call (pass the config from previous cell)
    covered_call_config=cc_config,
)

print(f"StrategyConfig: {len(config.ticker_universe)} symbols, cash=${config.starting_cash:,}")
print(f"  Entry: {config.entry_order_type}, steps={config.entry_max_steps}")
print(f"  Risk: delta_abs={config.enable_delta_absolute_stop}, vix_spike={config.enable_vix_spike_stop}")
print(f"  Paper: {config.paper_trading}")

In [ ]:
# ── Equity Screener Config (Phase 1) ────────────────────────────
# Load from named profile or customize inline:
# screener_config = EquityScreenerConfig.from_name("csp_bullish")  # from JSON profile

screener_config = EquityScreenerConfig(
    name="notebook_test",
    universe_source="screened_universe.json",

    # Technical indicators
    sma_periods=[8, 20, 50],
    rsi_period=14,
    rsi_lower=30,
    rsi_upper=70,
    bb_period=20,
    bb_std=1.0,
    sma_trend_lookback=3,
    history_days=60,

    # Filter toggles
    enable_sma8_check=True,
    enable_sma20_check=True,
    enable_sma50_check=True,
    enable_bb_upper_check=False,
    enable_band_check=True,
    enable_sma50_trend_check=True,
    enable_rsi_check=True,

    # Price filter
    share_price_max=None,              # e.g. 500.0 to exclude expensive stocks

    # Event toggles
    trade_during_earnings=False,
    trade_during_dividends=False,
    trade_during_fomc=False,
    max_dte=10,
)

print(f"EquityScreenerConfig: name={screener_config.name}")
print(f"  SMA periods: {screener_config.sma_periods}")
print(f"  RSI: [{screener_config.rsi_lower}, {screener_config.rsi_upper}]")
print(f"  Available JSON profiles: ", end="")
from pathlib import Path
profiles = [p.stem for p in (Path("equity_screener/configs")).glob("*.json")]
print(profiles or "none")

In [ ]:
# Cell 5: Connect to Alpaca + build data layer
alpaca = AlpacaClientManager(paper=config.paper_trading)
vix_fetcher = VixDataFetcher()
greeks_calc = GreeksCalculator()
data_manager = DataManager(alpaca, config)

current_vix = vix_fetcher.get_current_vix()
print(f"Alpaca connected (paper={alpaca.paper})")
print(f"VIX: {current_vix:.2f}")

In [ ]:
# Cell 6: Test build_execution_components factory
execution = ExecutionEngine(alpaca, config)

snapshot_fetcher = lambda symbols: _arun(
    data_manager.options_fetcher.get_option_snapshots, symbols
)
stepped, router = build_execution_components(
    execution=execution,
    config=config,
    snapshot_fetcher=snapshot_fetcher,
    vprint=print,
)

assert isinstance(stepped, SteppedOrderExecutor), "Expected SteppedOrderExecutor"
assert isinstance(router, ExitRouter), "Expected ExitRouter"
assert router.stepped_executor is stepped, "Router should reference the same executor"
print(f"build_execution_components: PASS")
print(f"  SteppedOrderExecutor: entry_max_steps={stepped.config.entry_max_steps}")
print(f"  ExitRouter: execution={type(router.execution).__name__}")

In [ ]:
# Cell 7: Test MarketSession
market_session = MarketSession(
    alpaca_manager=alpaca,
    vix_fetcher=vix_fetcher,
    vix_spike_multiplier=config.vix_spike_multiplier,
    vprint=print,
)

is_open = await market_session.is_market_open()
print(f"Market open: {is_open}")

vix_ref = await market_session.get_session_vix_reference()
print(f"Session VIX reference: {vix_ref:.2f}")

vix_stop = await market_session.check_global_vix_stop(current_vix)
print(f"VIX stop triggered (current={current_vix:.2f}): {vix_stop}")

print("\nMarketSession: PASS")

## CSP TradingLoop — Full Construction + Cycle

In [ ]:
# Cell 8: Build full TradingLoop (exercises all wiring)
scanner = StrategyScanner(
    config=config,
    equity_fetcher=data_manager.equity_fetcher,
    options_fetcher=data_manager.options_fetcher,
    greeks_calc=greeks_calc,
    equity_screener_config=screener_config,
)
risk_manager = RiskManager(config)
metadata = StrategyMetadataStore(path="strategy_metadata.json")

loop = TradingLoop(
    config=config,
    data_manager=data_manager,
    scanner=scanner,
    metadata_store=metadata,
    risk_manager=risk_manager,
    execution=execution,
    vix_fetcher=vix_fetcher,
    greeks_calc=greeks_calc,
    alpaca_manager=alpaca,
)

# Verify shared modules are wired correctly
assert isinstance(loop.stepped_executor, SteppedOrderExecutor), "stepped_executor wiring"
assert isinstance(loop.exit_router, ExitRouter), "exit_router wiring"
assert isinstance(loop.market_session, MarketSession), "market_session wiring"
assert loop.exit_router.stepped_executor is loop.stepped_executor, "router→executor link"

print("TradingLoop constructed — all shared modules wired correctly")

In [ ]:
# Cell 9: Run a single CSP cycle
summary = await loop.run_cycle()
print(f"\nCSP Cycle summary: {summary}")

## CC Manager — Full Construction + Cycle

In [ ]:
# Build CoveredCallManager
storage = build_storage_backend(config)
wheel_store = WheelPositionStore(path="wheel_positions_test.json", backend=storage)

cc_manager = CoveredCallManager(
    cc_config=cc_config,
    strategy_config=config,
    store=wheel_store,
    data_manager=data_manager,
    execution=execution,
    risk_manager=risk_manager,
    metadata_store=metadata,
    alpaca_manager=alpaca,
)

# Verify shared modules are wired correctly
assert isinstance(cc_manager.stepped_executor, SteppedOrderExecutor), "stepped_executor wiring"
assert isinstance(cc_manager.exit_router, ExitRouter), "exit_router wiring"
assert cc_manager.exit_router.stepped_executor is cc_manager.stepped_executor, "router->executor link"

print("CoveredCallManager constructed -- all shared modules wired correctly")

In [ ]:
# Cell 11: Run a CC cycle (will detect stock positions if any exist)
current_vix = vix_fetcher.get_current_vix()
cc_summary = await cc_manager.run_cycle(current_vix)
print(f"\nCC Cycle summary: {cc_summary}")

## Portfolio Status

In [ ]:
# Cell 12: Final portfolio check
print_portfolio_status(alpaca)

## Summary

If all cells above ran without errors:
- All imports work — shared utilities, extracted modules, screeners, core, and CC
- Shared utilities pass: `_arun`, `is_option_symbol`, `update_candidate_from_snapshot`
- `PositionProxy`, `ExecutionConfig`, `RiskConfig` adapters work
- `EquityScreenerConfig` loads (from inline or JSON profile)
- `build_execution_components` factory wires SteppedOrderExecutor + ExitRouter
- `MarketSession` constructs and fetches market state
- `TradingLoop` constructs with all Phase 1-4 shared modules and runs a cycle
- `CoveredCallManager` constructs with all Phase 1-4 shared modules and runs a cycle
- **All 4 phases of refactoring are validated end-to-end.**